# Scraper Torneos

Este notebook extrae todos los links de torneos desde la página https://atletismo.usplat.cl/torneos

## Importar librerías

In [1]:
import httpx
import lxml.html as lh
import pandas as pd
from tqdm import tqdm

## Configuración

In [2]:
URL_TORNEOS = "https://atletismo.usplat.cl/torneos"
BASE_URL = "https://atletismo.usplat.cl"

# Headers para la solicitud
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "es-ES,es;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
}

## Extraer links de torneos

In [3]:
# Crear cliente httpx
client = httpx.Client(headers=headers, follow_redirects=True)

# Hacer solicitud a la página de torneos
response = client.get(URL_TORNEOS)
print(f"Status code: {response.status_code}")

Status code: 200


In [4]:
# Parsear el HTML
tree = lh.fromstring(response.content)

# Extraer todos los links de torneos (elementos <a> con clase 'list-group-item')
# Los links están en formato: /torneo/nombre-torneo/
torneos_links = tree.xpath('//div[@class="list-group"]/a[@class="list-group-item"]/@href')
torneos_nombres = tree.xpath('//div[@class="list-group"]/a[@class="list-group-item"]/text()')

print(f"Total de torneos encontrados: {len(torneos_links)}")
print(f"\nPrimeros 5 torneos:")
for i in range(min(5, len(torneos_nombres))):
    print(f"{i+1}. {torneos_nombres[i]} - {torneos_links[i]}")

Total de torneos encontrados: 548

Primeros 5 torneos:
1. 10 km Facsa Castellón - /torneo/10-km-facsa-castellon/
2. 10 Km Valencia - /torneo/10-km-valencia/
3. Arena de Campeones Santo Tomás - /torneo/arena-de-campeones-santo-tomas/
4. Asociación Puerto Montt - /torneo/asociacion-puerto-montt/
5. Asociación Quinta Región - /torneo/region-valparaiso/


## Crear DataFrame con los datos

In [5]:
# Crear DataFrame
torneos_df = pd.DataFrame({
    "nombre_torneo": torneos_nombres,
    "url_relativa": torneos_links,
    "url_completa": [BASE_URL + link for link in torneos_links],
    "slug": [link.replace('/torneo/', '').replace('/', '') for link in torneos_links]
})

# Eliminar duplicados (algunos torneos pueden aparecer en múltiples pestañas)
torneos_df = torneos_df.drop_duplicates(subset=['slug']).reset_index(drop=True)

print(f"Total de torneos únicos: {len(torneos_df)}")
torneos_df.head(10)

Total de torneos únicos: 285


,nombre_torneo,url_relativa,url_completa,slug
0,10 km Facsa Castellón,/torneo/10-km-facsa-castellon/,https://atletismo.usplat.cl/torneo/10-km-facsa...,10-km-facsa-castellon
1,10 Km Valencia,/torneo/10-km-valencia/,https://atletismo.usplat.cl/torneo/10-km-valen...,10-km-valencia
2,Arena de Campeones Santo Tomás,/torneo/arena-de-campeones-santo-tomas/,https://atletismo.usplat.cl/torneo/arena-de-ca...,arena-de-campeones-santo-tomas
3,Asociación Puerto Montt,/torneo/asociacion-puerto-montt/,https://atletismo.usplat.cl/torneo/asociacion-...,asociacion-puerto-montt
4,Asociación Quinta Región,/torneo/region-valparaiso/,https://atletismo.usplat.cl/torneo/region-valp...,region-valparaiso
5,Campeonato Aniversario Club Atlético Volcanes,/torneo/campeonato-aniversario-club-atle/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-aniversario-club-atle
6,Campeonato Atlético Bajo Molle,/torneo/campeonato-atletico-bajo-monlle/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-atletico-bajo-monlle
7,Campeonato Atlético Formativo Pruebas Combinadas,/torneo/campeonato-formativo-PC/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-formativo-PC
8,Campeonato Atlético Gerardo Manzanares,/torneo/campeonato-atletico-gerardo-manz/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-atletico-gerardo-manz
9,Campeonato Club Atlético Volcanes,/torneo/campeonato-club-atletico-volcane/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-club-atletico-volcane


## Extraer categorías de torneos

Ahora vamos a identificar qué torneos pertenecen a cada categoría (Todos, Escolares, Federados)

In [6]:
# Extraer torneos por categoría

# Todos los torneos (tab 'Todos')
todos_links = tree.xpath('//div[@id="menu_todos"]//div[@class="list-group"]/a/@href')

# Torneos escolares
escolares_links = tree.xpath('//div[@id="menu_Escolares"]//div[@class="list-group"]/a/@href')

# Torneos federados
federados_links = tree.xpath('//div[@id="menu_Federados"]//div[@class="list-group"]/a/@href')

print(f"Torneos en 'Todos': {len(todos_links)}")
print(f"Torneos 'Escolares': {len(escolares_links)}")
print(f"Torneos 'Federados': {len(federados_links)}")

Torneos en 'Todos': 285
Torneos 'Escolares': 117
Torneos 'Federados': 146


In [7]:
# Crear columnas para categorías
torneos_df['es_escolar'] = torneos_df['url_relativa'].isin(escolares_links)
torneos_df['es_federado'] = torneos_df['url_relativa'].isin(federados_links)

# Determinar categoría principal
def obtener_categoria(row):
    if row['es_escolar'] and row['es_federado']:
        return 'Ambos'
    elif row['es_escolar']:
        return 'Escolar'
    elif row['es_federado']:
        return 'Federado'
    else:
        return 'Otro'

torneos_df['categoria'] = torneos_df.apply(obtener_categoria, axis=1)

# Ver distribución por categoría
print("\nDistribución de torneos por categoría:")
print(torneos_df['categoria'].value_counts())

torneos_df.head(10)


Distribución de torneos por categoría:
categoria
Federado    146
Escolar     117
Otro         22
Name: count, dtype: int64


,nombre_torneo,url_relativa,url_completa,slug,es_escolar,es_federado,categoria
0,10 km Facsa Castellón,/torneo/10-km-facsa-castellon/,https://atletismo.usplat.cl/torneo/10-km-facsa...,10-km-facsa-castellon,False,True,Federado
1,10 Km Valencia,/torneo/10-km-valencia/,https://atletismo.usplat.cl/torneo/10-km-valen...,10-km-valencia,False,True,Federado
2,Arena de Campeones Santo Tomás,/torneo/arena-de-campeones-santo-tomas/,https://atletismo.usplat.cl/torneo/arena-de-ca...,arena-de-campeones-santo-tomas,False,True,Federado
3,Asociación Puerto Montt,/torneo/asociacion-puerto-montt/,https://atletismo.usplat.cl/torneo/asociacion-...,asociacion-puerto-montt,False,True,Federado
4,Asociación Quinta Región,/torneo/region-valparaiso/,https://atletismo.usplat.cl/torneo/region-valp...,region-valparaiso,False,True,Federado
5,Campeonato Aniversario Club Atlético Volcanes,/torneo/campeonato-aniversario-club-atle/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-aniversario-club-atle,False,True,Federado
6,Campeonato Atlético Bajo Molle,/torneo/campeonato-atletico-bajo-monlle/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-atletico-bajo-monlle,False,True,Federado
7,Campeonato Atlético Formativo Pruebas Combinadas,/torneo/campeonato-formativo-PC/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-formativo-PC,False,True,Federado
8,Campeonato Atlético Gerardo Manzanares,/torneo/campeonato-atletico-gerardo-manz/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-atletico-gerardo-manz,False,True,Federado
9,Campeonato Club Atlético Volcanes,/torneo/campeonato-club-atletico-volcane/,https://atletismo.usplat.cl/torneo/campeonato-...,campeonato-club-atletico-volcane,False,True,Federado


## Guardar datos

In [8]:
# Guardar a CSV
torneos_df.to_csv('torneos.csv', index=False, encoding='utf-8-sig')
print(f"✓ Datos guardados en 'torneos.csv'")
print(f"Total de torneos: {len(torneos_df)}")

✓ Datos guardados en 'torneos.csv'
Total de torneos: 285


## Estadísticas adicionales

In [9]:
# Ver algunos ejemplos por categoría
print("\n=== TORNEOS ESCOLARES ===")
print(torneos_df[torneos_df['categoria'] == 'Escolar'][['nombre_torneo']].head(10))

print("\n=== TORNEOS FEDERADOS ===")
print(torneos_df[torneos_df['categoria'] == 'Federado'][['nombre_torneo']].head(10))

print("\n=== TORNEOS EN AMBAS CATEGORÍAS ===")
print(torneos_df[torneos_df['categoria'] == 'Ambos'][['nombre_torneo']].head(10))


=== TORNEOS ESCOLARES ===
                                      nombre_torneo
132  Campeonato Atlético Gerardo Manzanares Escolar
133                      Campeonato Atlético Junior
134                      Campeonato Atlético Senior
135                     Campeonato Lycee Claude Gay
136                            Campeonato Manquehue
137                           Control Foz do Iguazú
138              Copa Aniversario Ciudad de Iquique
139                        Copa Atlética Campanario
140                     Copa Atlético Limarí Ovalle
141               Copa Atletismo Escolar - Coquimbo

=== TORNEOS FEDERADOS ===
                                       nombre_torneo
0                     Arena de Campeones Santo Tomás
1                            Asociación Puerto Montt
2                           Asociación Quinta Región
3      Campeonato Aniversario Club Atlético Volcanes
4                     Campeonato Atlético Bajo Molle
5   Campeonato Atlético Formativo Pruebas Combinadas
6  

## Extraer HTML del primer torneo

In [10]:
# Obtener el primer torneo
primer_torneo = torneos_df.iloc[0]

print(f"Extrayendo HTML del primer torneo:")
print(f"Nombre: {primer_torneo['nombre_torneo']}")
print(f"URL: {primer_torneo['url_completa']}")
print(f"Slug: {primer_torneo['slug']}")
print(f"Categoría: {primer_torneo['categoria']}")
print(f"\nHaciendo solicitud...")

Extrayendo HTML del primer torneo:
Nombre: Arena de Campeones Santo Tomás
URL: https://atletismo.usplat.cl/torneo/arena-de-campeones-santo-tomas/
Slug: arena-de-campeones-santo-tomas
Categoría: Federado

Haciendo solicitud...


In [11]:
# Hacer solicitud al primer torneo
response_torneo = client.get(primer_torneo['url_completa'])
print(f"Status code: {response_torneo.status_code}")
print(f"Tamaño del contenido: {len(response_torneo.content)} bytes")

# Guardar el HTML en un archivo
nombre_archivo = f"torneo_{primer_torneo['slug']}.html"
with open(nombre_archivo, 'wb') as f:
    f.write(response_torneo.content)

print(f"\n✓ HTML guardado en: {nombre_archivo}")

Status code: 200
Tamaño del contenido: 55430 bytes

✓ HTML guardado en: torneo_arena-de-campeones-santo-tomas.html


In [ ]:
# Vista previa del HTML (primeros 1000 caracteres)
html_preview = response_torneo.text[:1000]
print("\\nVista previa del HTML:")
print("=" * 60)
print(html_preview)
print("=" * 60)
print(f"\\n... (total: {len(response_torneo.text)} caracteres)")